In [3]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Find the project root that contains the src folder
current = Path.cwd().resolve()

for p in [current] + list(current.parents):
    if (p / "src").exists():
        repo_root = p
        break
else:
    raise FileNotFoundError("Could not find a folder containing 'src'.")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("Current working directory:", current)
print("Added repo root:", repo_root)
print("src exists:", (repo_root / "src").exists())

from pathlib import Path
import numpy as np
import pandas as pd

import src.utils.pdata_io as pdio
from src.proc.extract_epoch_windows import load_epoch_windows

data_root, pdata_root, cc_data = pdio.load_project_context()

windows_df = load_epoch_windows(
    pdata_root=pdata_root,
    filename="behavior_epoch_windows.h5",
    key="windows/prepost_1s"
)

valid_windows = windows_df[windows_df["valid_window"]].copy()

print("All windows:", windows_df.shape)
print("Valid windows:", valid_windows.shape)

valid_windows.groupby(["phase", "epoch_name"]).size().reset_index(name="n_windows")

Current working directory: /home/nmldata2/ccaw/Python/notebooks
Added repo root: /home/nmldata2/ccaw/Python
src exists: True
/home/nmldata2/ccaw/Python
[LOADED] Project context: /mnt/pdata/Classical_Conditioning/_cache/project_context.pkl
[LOADED] Epoch windows: /mnt/pdata/Classical_Conditioning/_cache/behavior_epoch_windows.h5
[KEY] windows/prepost_1s
All windows: (42388, 16)
Valid windows: (38660, 16)


,phase,epoch_name,n_windows
0,air_training,air_off_post_1s,3380
1,air_training,air_off_pre_1s,3380
2,air_training,air_on_post_1s,3380
3,air_training,air_on_pre_1s,3380
4,habituation,LED_off_post_1s,2915
5,habituation,LED_off_pre_1s,2915
6,habituation,LED_on_post_1s,2915
7,habituation,LED_on_pre_1s,2915
8,tone_air_training,air_off_post_1s,1685
9,tone_air_training,air_off_pre_1s,1685


In [ ]:

from src.proc.behavior_metrics import compute_encoder_metrics_for_windows


encoder_epoch_df = compute_encoder_metrics_for_windows(
    valid_windows,
    speed_thresh=0.5
)

encoder_epoch_df.head()

,animal,date,phase,event_number,anchor_name,anchor_time_s,window_position,window_s,epoch_name,window_start_s,...,max_speed_net_cms,distance_path_cm,distance_net_cm,net_direction_bias,frac_stationary,frac_moving,frac_forward,frac_backward,frac_low_net_movement,dominant_locomotor_state
0,NML_04,2026_01_12,air_training,1,air_off,23.979000,post,1.0,air_off_post_1s,23.979000,...,14.074335,8.721061,8.721061,1.000000,0.0000,1.0000,1.0000,0.0000,0.0,forward
1,NML_04,2026_01_12,air_training,1,air_off,23.979000,pre,1.0,air_off_pre_1s,22.979000,...,13.571680,5.101946,4.850619,0.951624,0.0340,0.9660,0.8530,0.1130,0.0,forward
2,NML_04,2026_01_12,air_training,1,air_on,19.956400,post,1.0,air_on_post_1s,19.956400,...,0.753982,0.402124,-0.201062,-0.487133,0.5560,0.4440,0.0924,0.3516,0.0,stationary
3,NML_04,2026_01_12,air_training,1,air_on,19.956400,pre,1.0,air_on_pre_1s,18.956400,...,1.759292,0.753982,0.603186,0.799008,0.3630,0.6370,0.5644,0.0726,0.0,forward
4,NML_04,2026_01_12,air_training,2,air_off,41.279202,post,1.0,air_off_post_1s,41.279202,...,7.037168,3.066194,2.965663,0.982488,0.1328,0.8672,0.8480,0.0192,0.0,forward


In [8]:
cache_dir = Path(pdata_root) / "_cache"
cache_dir.mkdir(parents=True, exist_ok=True)

encoder_metrics_file = cache_dir / "behavior_epoch_metrics.h5"

encoder_epoch_df.to_hdf(
    encoder_metrics_file,
    key="encoder/prepost_1s_speedThresh_1cms",
    mode="w",
    format="table"
)

print("Saved:", encoder_metrics_file)

Saved: /mnt/pdata/Classical_Conditioning/_cache/behavior_epoch_metrics.h5


In [9]:
encoder_epoch_df.groupby(
    ["phase", "epoch_name"]
).size().reset_index(name="n_windows")

,phase,epoch_name,n_windows
0,air_training,air_off_post_1s,3380
1,air_training,air_off_pre_1s,3380
2,air_training,air_on_post_1s,3380
3,air_training,air_on_pre_1s,3380
4,habituation,LED_off_post_1s,2915
5,habituation,LED_off_pre_1s,2915
6,habituation,LED_on_post_1s,2915
7,habituation,LED_on_pre_1s,2915
8,tone_air_training,air_off_post_1s,1685
9,tone_air_training,air_off_pre_1s,1685


In [10]:
speed_summary = (
    encoder_epoch_df
    .groupby(["phase", "epoch_name"])
    .agg(
        mean_path_speed=("mean_speed_path_cms", "mean"),
        sem_path_speed=("mean_speed_path_cms", lambda x: x.std() / np.sqrt(len(x))),
        mean_net_speed=("mean_speed_net_cms", "mean"),
        n=("mean_speed_path_cms", "count"),
    )
    .reset_index()
)

speed_summary

,phase,epoch_name,mean_path_speed,sem_path_speed,mean_net_speed,n
0,air_training,air_off_post_1s,4.857098,0.182130,4.217457,3380
1,air_training,air_off_pre_1s,6.623915,0.055390,6.564985,3380
2,air_training,air_on_post_1s,3.061537,0.037536,2.721943,3380
3,air_training,air_on_pre_1s,1.500735,0.050661,1.442812,3380
4,habituation,LED_off_post_1s,1.683670,0.062997,1.477723,2915
5,habituation,LED_off_pre_1s,1.735510,0.065667,1.546038,2915
6,habituation,LED_on_post_1s,1.117324,0.039325,0.823163,2915
7,habituation,LED_on_pre_1s,1.613994,0.063275,1.399598,2915
8,tone_air_training,air_off_post_1s,3.397080,0.032434,2.938223,1685
9,tone_air_training,air_off_pre_1s,5.609373,0.053521,5.523539,1685


In [11]:
state_summary = (
    encoder_epoch_df
    .groupby(["phase", "epoch_name"])
    .agg(
        frac_stationary=("frac_stationary", "mean"),
        frac_moving=("frac_moving", "mean"),
        frac_forward=("frac_forward", "mean"),
        frac_backward=("frac_backward", "mean"),
        frac_low_net=("frac_low_net_movement", "mean"),
        n=("frac_stationary", "count"),
    )
    .reset_index()
)

state_summary

,phase,epoch_name,frac_stationary,frac_moving,frac_forward,frac_backward,frac_low_net,n
0,air_training,air_off_post_1s,0.063616,0.936384,0.852149,0.084235,0.0,3380
1,air_training,air_off_pre_1s,0.016974,0.983026,0.967295,0.015730,0.0,3380
2,air_training,air_on_post_1s,0.206528,0.793472,0.678352,0.115120,0.0,3380
3,air_training,air_on_pre_1s,0.707988,0.292012,0.272948,0.019063,0.0,3380
4,habituation,LED_off_post_1s,0.680382,0.319618,0.256137,0.063481,0.0,2915
5,habituation,LED_off_pre_1s,0.688589,0.311411,0.256306,0.055105,0.0,2915
6,habituation,LED_on_post_1s,0.704919,0.295081,0.219094,0.075988,0.0,2915
7,habituation,LED_on_pre_1s,0.703182,0.296818,0.236503,0.060314,0.0,2915
8,tone_air_training,air_off_post_1s,0.082443,0.917557,0.800552,0.117006,0.0,1685
9,tone_air_training,air_off_pre_1s,0.023539,0.976461,0.955140,0.021322,0.0,1685
